# 31 — Revision Triggers

## When should a system revise instead of reset?

Notebook 29 compared budgets.

Notebook 31 asks:

> Can a system detect when revision is needed before reset becomes necessary?

Outputs:

```text
results/31_revision_triggers.csv
figures/31_revision_triggers.png
```

No model downloads. No GPU. No pruning jobs.

In [ ]:
# Cell 1 — locate or clone repo

from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/thinkthoughts/pruning-rml.git"
REPO_NAME = "pruning-rml"

cwd = Path.cwd().resolve()

if cwd.name == REPO_NAME:
    repo = cwd
elif cwd.name == "notebooks" and cwd.parent.name == REPO_NAME:
    repo = cwd.parent
elif Path("/content/pruning-rml").exists():
    repo = Path("/content/pruning-rml")
elif (cwd / REPO_NAME).exists():
    repo = cwd / REPO_NAME
else:
    target = Path("/content") if Path("/content").exists() else cwd
    os.chdir(target)
    subprocess.run(["git", "clone", REPO_URL], check=True)
    repo = target / REPO_NAME

os.chdir(repo)

src = repo / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

print("repo:", repo)
print("src :", src)
print("src exists:", src.exists())

In [ ]:
# Cell 2 — ensure helpers

pkg = repo / "src" / "pruning_rml"
pkg.mkdir(parents=True, exist_ok=True)

(pkg / "__init__.py").write_text('__version__ = "0.1.0"\n', encoding="utf-8")

(pkg / "paths.py").write_text('''from pathlib import Path

def ensure_dirs(root, names=("results", "figures", "reports")):
    root = Path(root)
    outputs = {}
    for name in names:
        path = root / name
        path.mkdir(parents=True, exist_ok=True)
        outputs[name] = path
    return outputs
''', encoding="utf-8")

(pkg / "triggers.py").write_text('''def trigger_score(staleness, drift, cost_pressure, residue_strength):
    return staleness + drift + cost_pressure - residue_strength

def trigger_action(score):
    if score <= 1:
        return "continue"
    if score <= 3:
        return "revise"
    return "reset_or_major_revision"
''', encoding="utf-8")

for module_name in [
    "pruning_rml",
    "pruning_rml.paths",
    "pruning_rml.triggers",
]:
    if module_name in sys.modules:
        del sys.modules[module_name]

print("helpers ready")
print("package files:", sorted(p.name for p in pkg.iterdir()))

In [ ]:
# Cell 3 — imports and output folders

import pandas as pd
import matplotlib.pyplot as plt

from pruning_rml.paths import ensure_dirs
from pruning_rml.triggers import trigger_score, trigger_action

outputs = ensure_dirs(repo)
results = outputs["results"]
figures = outputs["figures"]

print("results:", results)
print("figures:", figures)

## Trigger schema

A revision trigger combines four signals:

```text
staleness
+ drift
+ cost pressure
- residue strength
```

Interpretation:

- high staleness means accumulated structure may be outdated
- high drift means behavior is moving away from target constraints
- high cost pressure means the model may need compression
- high residue strength means useful structure still remains

If useful residues remain, revision is preferred over reset.

In [ ]:
# Cell 4 — build trigger table

rows = [
    {
        "condition": "stable inherited structure",
        "staleness": 0,
        "drift": 0,
        "cost_pressure": 1,
        "residue_strength": 2,
    },
    {
        "condition": "mild cost pressure",
        "staleness": 1,
        "drift": 0,
        "cost_pressure": 2,
        "residue_strength": 2,
    },
    {
        "condition": "capability drift",
        "staleness": 1,
        "drift": 2,
        "cost_pressure": 1,
        "residue_strength": 1,
    },
    {
        "condition": "stale accumulated structure",
        "staleness": 3,
        "drift": 1,
        "cost_pressure": 2,
        "residue_strength": 1,
    },
    {
        "condition": "low residue after compression",
        "staleness": 2,
        "drift": 2,
        "cost_pressure": 2,
        "residue_strength": 0,
    },
]

df = pd.DataFrame(rows)
df["trigger_score"] = df.apply(
    lambda row: trigger_score(
        row["staleness"],
        row["drift"],
        row["cost_pressure"],
        row["residue_strength"],
    ),
    axis=1,
)
df["action"] = df["trigger_score"].apply(trigger_action)

df

In [ ]:
# Cell 5 — save CSV

csv_path = results / "31_revision_triggers.csv"
df.to_csv(csv_path, index=False)

print("saved:", csv_path)
print("exists:", csv_path.exists())

## Visual summary

The plot shows when the system moves from continuing to revising to major reset.

Working hypothesis:

> pruning becomes useful when cost pressure and drift rise while useful residues still remain.

In [ ]:
# Cell 6 — save figure

fig_path = figures / "31_revision_triggers.png"

ax = df.plot.bar(
    x="condition",
    y="trigger_score",
    legend=False,
    figsize=(10, 5),
)

ax.set_title("Revision triggers")
ax.set_xlabel("Condition")
ax.set_ylabel("Trigger score")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(fig_path, dpi=160)
plt.show()

print("saved:", fig_path)
print("exists:", fig_path.exists())

In [ ]:
# Cell 7 — verify outputs

expected = [
    results / "31_revision_triggers.csv",
    figures / "31_revision_triggers.png",
]

for path in expected:
    print(path, "exists:", path.exists())

## Result

Notebook 31 creates:

```text
results/31_revision_triggers.csv
figures/31_revision_triggers.png
```

Next notebook:

```text
37_rml_guided_pruning.ipynb
```

Next question:

> Can residue and trigger signals guide pruning choices?